# 11 · Agents and workflows

**Goal:** handle tasks that one request can't do well, using either a **workflow** (you decide the steps) or an **agent** (Claude decides the steps), and know which to choose.

| | |
|---|---|
| **You will learn** | Chaining, routing, parallelization, evaluator-optimizer, agents with tools, environment inspection, orchestrator-workers, and when to use each |
| **Prerequisites** | [Notebook 02](02_structured_outputs.ipynb) for structured outputs, and [notebook 05](05_tool_use.ipynb) for the tool loop |
| **Run time** | About 3 minutes |
| **Cost** | About $0.08 when I ran it. This notebook stops itself at $0.35 |
| **Paid cells** | Sections 2 to 8. Sections 1 and 9 are free |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

Several demos use the fictional company report in `data/halvora_report.md`.

## Setup

The same setup as the earlier notebooks, with one addition: `track()` uses a **lock**. Section 4 runs several requests at the same time, and the lock stops two of them from updating `spent` at the same moment. Run the cells from top to bottom.

In [8]:
import ast
import json
import operator
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from statistics import mean
from typing import Literal

from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

print("Ready. Using model:", MODEL)

Ready. Using model: claude-sonnet-5-5


In [9]:
INPUT_PRICE = 2.00    # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00  # dollars per 1 million output tokens (Sonnet 5.5)
BUDGET_USD = 0.35     # the most this notebook should spend

spent = 0.0
spend_lock = threading.Lock()  # lets only one request at a time update `spent`


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000
    with spend_lock:
        spent += cost
        total = spent
    print(f"[input={input_tokens:,} output={output_tokens:,} cost=${cost:.5f} | spent ${total:.4f} of ${BUDGET_USD:.2f}]")
    if total > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${total:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def ask(prompt, system=None, max_tokens=1000):
    """Send one prompt and return the text of the answer."""
    params = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "output_config": {"effort": "low"},
        "messages": [{"role": "user", "content": prompt}],
    }
    if system:
        params["system"] = system
    return text_from_message(track(client.messages.create(**params)))


def ask_structured(prompt, output_format, max_tokens=1000):
    """Send one prompt and return a validated object (structured outputs, notebook 02)."""
    response = track(client.messages.parse(
        model=MODEL,
        max_tokens=max_tokens,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
        output_format=output_format,
    ))
    return response.parsed_output

In [10]:
# The fictional Halvora report, split into sections by their "## " headings.
report = open("data/halvora_report.md", encoding="utf-8").read()

sections = {}
for chunk in re.split(r"\n(?=## )", report)[1:]:            # [1:] skips the title block
    heading = chunk.splitlines()[0].lstrip("# ").strip()      # "Section 2: Software Engineering"
    title = heading.split(":", 1)[-1].strip()                 # "Software Engineering"
    sections[title] = chunk.strip()

print(f"{len(sections)} sections:", list(sections))
assert "Software Engineering" in sections and "Human Resources" in sections

15 sections: ['Executive Summary', 'Table of Contents', 'Methodology', 'Medical Research', 'Software Engineering', 'Financial Analysis', 'Product Development', 'Legal Developments', 'Human Resources', 'Marketing and Brand', 'Supply Chain and Operations', 'Sustainability', 'Cybersecurity Analysis', 'Research Partnerships', 'Future Outlook']


## 1. Workflows and agents (free)

Both are ways to solve a task that one request can't do well. The difference is **who decides the steps**.

| | Workflow | Agent |
|---|---|---|
| **Who decides the steps** | You, in code | Claude, by choosing tools |
| **Best for** | Tasks you can map out in advance | Varied tasks you can't predict |
| **Reliability** | Higher: each step is small and focused | Lower: Claude can take a wrong turn |
| **Testing** | Easier: you know every step | Harder: the steps change from run to run |
| **Cost** | Predictable | Varies with how many steps Claude takes |
| **Flexibility** | Limited to the steps you wrote | Can handle tasks you didn't plan for |

A workflow is a recipe: you write the steps, and Claude does each one. An agent is a chef with a stocked kitchen: you say what dish you want, and the chef decides how to make it.

Sections 2 to 5 and 8 are workflow patterns. Sections 6 and 7 are about agents.

## 2. Chaining: one step after another (paid)

**Chaining** splits a task into steps that run in order, each one using the output of the last. You can also run ordinary code between the steps.

A classic use: long prompts full of rules, where Claude may quietly break a rule or two. Instead of fighting with one huge prompt:

1. **Generate** a draft with the rules.
2. **Check** the draft with plain code. This step is free.
3. **Revise**, only if the check found problems, in a separate call that does nothing but fix those problems.

Each call has one job, which is easier to get right than many jobs at once.

In [ ]:
RULES = """
* Write about 150 words for software developers.
* Use a professional, technical tone.
* Do not use emojis.
* Do not use exclamation marks.
* Do not mention that you are an AI or a language model.
"""


def find_problems(text):
    """Code check for the rules a program can test. Returns a list of problems."""
    problems = []
    # Check the writing, not the code samples: Python's != is not an exclamation mark.
    prose = re.sub(r"```.*?```|`[^`]*`", "", text, flags=re.DOTALL)
    if re.search(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", prose):
        problems.append("contains emojis")
    if "!" in prose:
        problems.append("contains exclamation marks")
    if re.search(r"\b(as an ai|i am an ai|i'm an ai|language model)\b", prose, re.IGNORECASE):
        problems.append("mentions being an AI")
    words = len(text.split())
    if not 100 <= words <= 220:
        problems.append(f"has {words} words, but should be about 150")
    return problems


# Step 1: generate.
draft = ask(f"Write a short article explaining Python list comprehensions.\n\nRules:{RULES}")
problems = find_problems(draft)
print("\nDraft problems:", problems or "none")

# Step 2: revise, but only if the free code check found something.
if problems:
    final = ask(f"""Revise the article below. Fix only these problems, and change nothing else:
{chr(10).join('- ' + p for p in problems)}

<article>
{draft}
</article>

Reply with the revised article only.""")
else:
    final = draft
    print("The draft already passed every check, so the revise step was skipped.")

final_problems = find_problems(final)
print("\nFinal problems:", final_problems or "none")
print("\n" + final)

**What to notice**

- `find_problems` is ordinary code running between two Claude calls. Chains can mix code and model steps freely.
- **The check removes code samples before it looks for problems.** My first version didn't, so it read Python's `!=` operator as an exclamation mark. It flagged a problem that wasn't there, paid for a revise call that couldn't fix it, and still reported a failure. **Code checks can be wrong too**, so test them, as notebook 03 did with its regex grader.
- If the draft already passed, the chain skipped step 2 and saved a call. A chain doesn't have to run every step every time.
- Use chaining when **steps depend on each other**, so they must run in order.

## 3. Routing: send each request to the right specialist (paid)

Different kinds of request need different handling. A video script about Python functions should teach. One about surfing should entertain. One prompt can't be ideal for both.

**Routing** classifies the request first, then sends it to **exactly one** specialised prompt. We classify with structured outputs, so the category can only be one of the values we allow.

In [12]:
class Route(BaseModel):
    category: Literal["educational", "entertainment", "review"]


TEMPLATES = {
    "educational": "Write a 3-sentence script for a short educational video about {topic}. "
                   "Explain one idea clearly, with a relatable example.",
    "entertainment": "Write a 3-sentence script for a short, high-energy entertainment video about {topic}. "
                     "Make it vivid and fun.",
    "review": "Write a 3-sentence script for a short review video about {topic}. "
              "Give one clear strength, one clear weakness, and a verdict.",
}


def route_and_write(topic):
    route = ask_structured(f"Which kind of short video suits this topic best?\n\n<topic>{topic}</topic>", Route)
    print(f"Topic: {topic} -> category: {route.category}")
    script = ask(TEMPLATES[route.category].format(topic=topic), max_tokens=500)
    print(script, "\n")
    return route.category


for topic in ["Python functions", "Surfing at sunrise", "A new budget smartphone's camera"]:
    category = route_and_write(topic)
    assert category in TEMPLATES

[input=268 output=11 cost=$0.00065 | spent $0.0092 of $0.35]
Topic: Python functions -> category: educational
[input=47 output=212 cost=$0.00221 | spent $0.0114 of $0.35]
**Video Script: "Python Functions: Your Reusable Recipe"**

**[Sentence 1]** A function is a reusable block of code that you define once and run whenever you need it, just like a recipe you write down once and cook anytime.

**[Sentence 2]** For example, instead of retyping the same greeting code over and over, you can write `def greet(name): return f"Hello, {name}!"` and then simply call `greet("Maya")` to get "Hello, Maya!"

**[Sentence 3]** Because the name is a placeholder the function fills in each time, one small function can handle a thousand different people, saving you time and keeping your code clean. 

[input=273 output=13 cost=$0.00068 | spent $0.0121 of $0.35]
Topic: Surfing at sunrise -> category: entertainment
[input=52 output=220 cost=$0.00230 | spent $0.0144 of $0.35]
**[SCENE: Golden light spills ove

**What to notice**

- The router call is small and cheap. Each specialist prompt can then be tuned and tested on its own.
- Routing sends a request to **one** path. Parallelization, next, sends it to **all** paths at once.

## 4. Parallelization: several focused checks at once (paid)

Asking Claude to weigh many criteria in one request makes it juggle them all at once. **Parallelization** splits the job into independent sub-tasks, runs them at the same time, then combines the results:

```
              ┌─▶ judge on customer value ─┐
 the idea ────┼─▶ judge on cost ───────────┼──▶ combine into one recommendation
              ├─▶ judge on risk ───────────┤
              └─▶ judge on effort ─────────┘
```

Each judge has one narrow job. You can improve one judge's prompt without touching the others, and adding a criterion means adding one more call.

`ThreadPoolExecutor` runs the four calls at the same time. They all add to `spent`, which is why `track()` uses a lock: it makes each update wait its turn, so no update is lost.

In [ ]:
IDEA = "Add an AI assistant to Halvora's reporting product that answers questions about a customer's own dashboards."

CRITERIA = {
    "customer value": "How much would customers benefit from this?",
    "cost": "How expensive would this be to build and run? A high score means cheap.",
    "risk": "How risky is this for security, privacy and accuracy? A high score means low risk.",
    "effort": "How quickly could a small team ship a first version? A high score means quick.",
}


class Judgement(BaseModel):
    score: int = Field(ge=1, le=5, description="1 is very poor, 5 is excellent")
    reason: str


def judge(criterion):
    prompt = f"""Judge this product idea on ONE criterion only: {criterion}.
{CRITERIA[criterion]}

<idea>{IDEA}</idea>

Give a score from 1 to 5 and a one-sentence reason."""
    return criterion, ask_structured(prompt, Judgement)


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    judgements = dict(pool.map(judge, CRITERIA))
print(f"\nFour judges finished in {time.perf_counter() - start:.1f}s, running at the same time.\n")

for criterion, judgement in judgements.items():
    print(f"{judgement.score}/5  {criterion:<15} {judgement.reason}")
print(f"\nAverage: {mean(j.score for j in judgements.values()):.2f} / 5")

# The combine step: one more call reads all four judgements.
summary = "\n".join(f"- {c} ({CRITERIA[c]}): {j.score}/5. {j.reason}" for c, j in judgements.items())
recommendation = ask(f"""Here are four separate judgements of a product idea.

<idea>{IDEA}</idea>

<judgements>
{summary}
</judgements>

In two sentences, recommend whether to build it and what to watch out for.""", max_tokens=400)
print("\nRecommendation:", recommendation)

assert set(judgements) == set(CRITERIA)

**What to notice**

- The four judges ran at the same time, so the wall-clock time was close to one call, not four. The **cost** is still four calls.
- **The combine step gets each criterion's definition, not just its score.** In my first run it saw only "effort: 4/5", and Claude pointed out it couldn't tell whether 4 meant easy or hard. The aggregator only knows what you pass it.
- The sub-tasks don't need to be identical. Each judge could have its own prompt, tools or even model.
- Use parallelization when the sub-tasks are **independent**. If one needs another's result, use chaining.

## 5. Evaluator-optimizer: draft, check, improve (paid)

A **producer** writes something, and an **evaluator** checks it. If it fails, the feedback goes back to the producer, and the loop repeats until the work passes or a round limit is hit.

```
 producer ──▶ draft ──▶ evaluator ──▶ passes? ──yes──▶ done
     ▲                                   │
     └────────── feedback ◀──── no ──────┘
```

Our task: a board summary of the Software Engineering section. Every rule here can be checked by code, so the evaluator is free and always gives the same answer. For qualities code can't check, use a model grader from notebook 03 as the evaluator instead.

In [14]:
SOURCE = sections["Software Engineering"]


def evaluate_summary(summary):
    """The evaluator: returns a list of failed checks. An empty list means accepted."""
    failures = []
    bullets = [line for line in summary.splitlines() if line.startswith("- ")]
    if len(bullets) != 3:
        failures.append(f"needs exactly 3 bullets starting with '- ', found {len(bullets)}")
    for number, bullet in enumerate(bullets, start=1):
        words = len(bullet[2:].split())
        if words > 20:
            failures.append(f"bullet {number} has {words} words, the limit is 20")
        figures = re.findall(r"\d[\d,.]*%?", bullet)
        if not figures:
            failures.append(f"bullet {number} has no number from the source")
        for figure in figures:
            if figure.rstrip(".,") not in SOURCE:
                failures.append(f"bullet {number} uses '{figure}', which is not in the source")
    return failures


def produce_summary(previous=None, feedback=None):
    """The producer: writes a first draft, or fixes the previous one using the feedback."""
    prompt = f"""Summarise this section for Halvora's board.

<section>
{SOURCE}
</section>

Rules: exactly 3 bullets, each starting with "- ", each at most 20 words,
each containing at least one number copied exactly from the section. Reply with the bullets only."""
    if previous:
        prompt += f"""

Your previous draft failed these checks:
{chr(10).join('- ' + f for f in feedback)}

<previous_draft>
{previous}
</previous_draft>

Write a corrected version."""
    return ask(prompt, max_tokens=500)


MAX_ROUNDS = 3
summary, failures = None, None
for round_number in range(1, MAX_ROUNDS + 1):
    summary = produce_summary(summary, failures)
    failures = evaluate_summary(summary)
    print(f"Round {round_number}: {'ACCEPTED' if not failures else failures}\n")
    if not failures:
        break

print(summary)
assert round_number <= MAX_ROUNDS

[input=475 output=159 cost=$0.00254 | spent $0.0289 of $0.35]
Round 1: ['bullet 3 has 21 words, the limit is 20']

[input=691 output=156 cost=$0.00294 | spent $0.0318 of $0.35]
Round 2: ACCEPTED

- Project Phoenix, a 2-year stability programme, added circuit breakers, failover message queues, and canary releases reaching 2% of traffic first.
- Bugs fixed rose to 1,284 from 1,037; pre-release detection improved from 61% to 74%; outages fell from 19 to 7.
- Recovery time fell from 52 to 18 minutes; INC-2023-Q4-011's flawed library was patched across 140 services within 48 hours.


**What to notice**

- If round 1 passed, the loop stopped straight away. The evaluator only costs extra rounds when the work needs them.
- `MAX_ROUNDS` matters. Without a limit, a producer that keeps failing would loop, and spend, forever.
- The number check catches **invented figures**: every number in a bullet must appear in the source text.

## 6. Agents: give Claude tools and a goal (paid)

In a workflow, you write the steps. With an **agent**, you give Claude a goal and some tools, and Claude decides which tools to call and in what order. It's the tool loop from notebook 05, with an open goal.

The tools are small and general: list the sections, read one section, do arithmetic. None of them is built for our particular question. Claude combines them. General tools make an agent useful for many questions you never planned for.

> **About `calculate`:** never pass model-written text to Python's `eval()`, because it would run any code at all. Our calculator reads the expression with the `ast` module and allows only numbers and `+ - * / ( )`. Anything else, including `**`, is refused.

In [15]:
def list_sections():
    return list(sections)


def read_section(title):
    if title not in sections:
        raise ValueError(f"No section called {title!r}. Call list_sections to see the titles.")
    return sections[title]


ALLOWED_OPERATORS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
                     ast.Div: operator.truediv, ast.USub: operator.neg}


def calculate(expression):
    """Safely evaluate arithmetic such as '(1730 - 1420) / 1420 * 100'. Nothing else is allowed."""
    def evaluate(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_OPERATORS:
            return ALLOWED_OPERATORS[type(node.op)](evaluate(node.left), evaluate(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_OPERATORS:
            return ALLOWED_OPERATORS[type(node.op)](evaluate(node.operand))
        raise ValueError("Only numbers and + - * / ( ) are allowed.")
    return round(evaluate(ast.parse(expression, mode="eval").body), 4)


agent_tools = [
    {"name": "list_sections", "description": "List the titles of every section in Halvora's 2023 annual report.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "read_section", "description": "Read the full text of one section of the report. "
                                            "Call list_sections first if you don't know the exact title.",
     "input_schema": {"type": "object", "properties": {"title": {"type": "string", "description": "The exact section title"}},
                      "required": ["title"]}},
    {"name": "calculate", "description": "Evaluate an arithmetic expression such as '(1730 - 1420) / 1420 * 100'. "
                                         "Use this for any calculation instead of doing it yourself.",
     "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
]

TOOL_FUNCTIONS = {"list_sections": list_sections, "read_section": read_section, "calculate": calculate}


def run_agent(goal, tools, system=None, max_turns=10):
    """The tool loop from notebook 05. Returns the final answer and the tools Claude called."""
    messages = [{"role": "user", "content": goal}]
    steps = []
    for turn in range(1, max_turns + 1):
        params = {"model": MODEL, "max_tokens": 1500, "output_config": {"effort": "low"},
                  "messages": messages, "tools": tools}
        if system:
            params["system"] = system
        response = track(client.messages.create(**params))
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            return text_from_message(response), steps

        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            steps.append(block.name)
            try:
                output = TOOL_FUNCTIONS[block.name](**block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)})
            except Exception as error:
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": f"Error: {error}", "is_error": True})
            print(f"   turn {turn}: {block.name}({json.dumps(block.input)[:70]})")
        messages.append({"role": "user", "content": results})
    raise RuntimeError(f"No final answer after {max_turns} turns.")


answer, steps = run_agent(
    "Using Halvora's annual report, which grew faster in 2023 in percentage terms: revenue or headcount? "
    "Show the two percentages.",
    agent_tools,
)
print("\nSteps Claude chose:", steps)
print("\nAnswer:", answer)

print("\nChecked by our own code: headcount grew", f"{(1730 - 1420) / 1420 * 100:.1f}%", "and revenue grew 14%.")
assert steps

[input=623 output=27 cost=$0.00152 | spent $0.0333 of $0.35]
   turn 1: list_sections({})
[input=782 output=101 cost=$0.00257 | spent $0.0359 of $0.35]
   turn 2: read_section({"title": "Financial Analysis"})
   turn 2: read_section({"title": "Human Resources"})
[input=1,583 output=64 cost=$0.00381 | spent $0.0397 of $0.35]
   turn 3: calculate({"expression": "(1730 - 1420) / 1420 * 100"})
[input=1,655 output=127 cost=$0.00458 | spent $0.0443 of $0.35]

Steps Claude chose: ['list_sections', 'read_section', 'read_section', 'calculate']

Answer: **Headcount grew faster in 2023.**

- **Headcount:** It rose from 1,420 to 1,730 employees, which is about **21.8%** growth.
- **Revenue:** It grew **14%** to $412 million. The report states this percentage directly, and I didn't calculate it myself.

Headcount growth was about 7.8 percentage points higher than revenue growth.

Checked by our own code: headcount grew 21.8% and revenue grew 14%.


**What to notice**

- **You never told Claude which sections to read or what to calculate.** It found the revenue figure and the headcount figures itself, then used `calculate`.
- Run it again and the steps may differ, for example reading a different section first. That flexibility is the strength of an agent, and also why agents are harder to test.
- `max_turns` and `track()` keep a wandering agent from spending without limit.

## 7. Environment inspection: let the agent check its own work (paid)

An agent acts blindly unless it can **see the results of its actions**. Two habits make agents far more reliable:

- **Read before you write.** Look at the current state before changing it.
- **Check after you act.** Verify the change did what you wanted.

Here the agent edits a section and must meet a word limit. We give it `count_words`, so it can measure its result instead of guessing, and the system prompt asks for both habits. The edit works on a copy of the report in memory, so the file on disk doesn't change.

In [ ]:
def replace_text(title, old_text, new_text):
    current = read_section(title)
    if current.count(old_text) != 1:
        raise ValueError("old_text must appear exactly once in the section. Read the section and copy it exactly.")
    sections[title] = current.replace(old_text, new_text)
    return "Replaced. Read the section again to check the result."


def count_words(text):
    return len(text.split())


edit_tools = agent_tools[:2] + [
    {"name": "replace_text", "description": "Replace one exact piece of text in a section with new text.",
     "input_schema": {"type": "object", "properties": {
         "title": {"type": "string"},
         "old_text": {"type": "string", "description": "Text copied exactly from the section, appearing once"},
         "new_text": {"type": "string"}}, "required": ["title", "old_text", "new_text"]}},
    {"name": "count_words", "description": "Count the words in a piece of text. Use this to check a word limit.",
     "input_schema": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}},
]
TOOL_FUNCTIONS.update({"replace_text": replace_text, "count_words": count_words})

original = sections["Sustainability"]
first_paragraph = original.split("\n\n")[1]
print(f"First paragraph before: {count_words(first_paragraph)} words\n")

answer, steps = run_agent(
    "Shorten the first paragraph of the Sustainability section to at most 25 words. "
    "Keep the figures 18% and 64%.",
    edit_tools,
    system= "Always read a section before you edit it. After every edit, read the section again "
           "and use count_words to check the result before you say you are done.",
)
print("\nSteps Claude chose:", steps)
print("Agent says:", answer)

new_first_paragraph = sections["Sustainability"].split("\n\n")[1]
print(f"\nOur own check: {count_words(new_first_paragraph)} words, "
      f"18% kept: {'18%' in new_first_paragraph}, 64% kept: {'64%' in new_first_paragraph}")
print(new_first_paragraph)

sections["Sustainability"] = original  # put the in-memory report back as it was

First paragraph before: 30 words

[input=759 output=27 cost=$0.00179 | spent $0.0461 of $0.35]
   turn 1: list_sections({})
[input=918 output=51 cost=$0.00235 | spent $0.0484 of $0.35]
   turn 2: read_section({"title": "Sustainability"})
[input=1,220 output=359 cost=$0.00603 | spent $0.0544 of $0.35]
   turn 3: replace_text({"title": "Sustainability", "old_text": "We cut data-centre energy use)
   turn 3: read_section({"title": "Sustainability"})
   turn 3: count_words({"text": "We cut data-centre energy use by 18% through server upgrades)
[input=1,917 output=161 cost=$0.00544 | spent $0.0599 of $0.35]

Steps Claude chose: ['list_sections', 'read_section', 'replace_text', 'read_section', 'count_words']
Agent says: I shortened the first paragraph of the Sustainability section to 24 words, under the 25-word limit. I re-read the section after the edit and counted the words in the new paragraph.

It now reads: "We cut data-centre energy use by 18% through server upgrades, better cooling, a

**What to notice**

- Look at the order of the steps: a `read_section` before the edit, then a read and a `count_words` after it. That's the agent inspecting its environment.
- **Our own check at the end** doesn't trust the agent's word. It measures the result itself. Do this in your own projects too.
- `replace_text` refuses an `old_text` that doesn't match exactly once. A clear error message tells the agent how to recover, usually by reading the section again.

## 8. Orchestrator-workers: Claude plans, workers do the parts (paid)

This pattern is in Anthropic's guide to building effective agents, but not in the course. It sits between a workflow and an agent:

1. An **orchestrator** call breaks the task into sub-tasks. Unlike parallelization, **Claude decides the sub-tasks at run time**.
2. **Workers** handle the sub-tasks, here in parallel.
3. A final call **combines** the results.

It suits tasks where you can't know the pieces in advance, such as a briefing whose topics depend on the question.

In [17]:
class SubTask(BaseModel):
    section: Literal[tuple(sections)]   # must be a real section title
    question: str


class Plan(BaseModel):
    subtasks: list[SubTask] = Field(description="2 to 4 sub-tasks")


request = "Prepare a short board briefing on how Halvora's people, security and money changed in 2023."

# 1. The orchestrator plans.
plan = ask_structured(f"""Break this request into 2 to 4 sub-tasks. Each one answers one question from one report section.

<request>{request}</request>
<sections>{list(sections)}</sections>""", Plan)
for task in plan.subtasks:
    print(f"   sub-task: [{task.section}] {task.question}")


# 2. Workers answer each sub-task from its own section, at the same time.
def worker(task):
    return ask(f"""Answer in one sentence, using only this section.

<section>{sections[task.section]}</section>
<question>{task.question}</question>""", max_tokens=300)


with ThreadPoolExecutor(max_workers=4) as pool:
    answers = list(pool.map(worker, plan.subtasks))

# 3. Combine.
notes = "\n".join(f"- {a}" for a in answers)
briefing = ask(f"Combine these notes into one short paragraph for the board.\n\n<notes>\n{notes}\n</notes>", max_tokens=400)
print("\nBriefing:", briefing)

assert len(plan.subtasks) >= 1

[input=787 output=237 cost=$0.00394 | spent $0.0638 of $0.35]
   sub-task: [Human Resources] How did Halvora's workforce change in 2023, including headcount, hiring, attrition, and any major organizational or talent changes?
   sub-task: [Cybersecurity Analysis] How did Halvora's security posture change in 2023, including notable incidents, vulnerabilities, investments, and risk exposure?
   sub-task: [Financial Analysis] How did Halvora's financial performance change in 2023, including revenue, profitability, cash position, and major spending shifts?
   sub-task: [Executive Summary] What are the key takeaways for the board on how Halvora's people, security, and finances changed in 2023?
[input=352 output=108 cost=$0.00178 | spent $0.0656 of $0.35]
[input=332 output=195 cost=$0.00261 | spent $0.0682 of $0.35]
[input=432 output=149 cost=$0.00235 | spent $0.0706 of $0.35]
[input=453 output=297 cost=$0.00388 | spent $0.0745 of $0.35]
[input=784 output=400 cost=$0.00557 | spent $0.0800 of 

**What to notice**

- `Literal[tuple(sections)]` limits each sub-task to a real section title, so a worker can never be sent to a section that doesn't exist.
- The plan changes with the request. Ask about sustainability and research, and you get different sub-tasks with no code changes.

## 9. Which one should you use? (free)

| Pattern | Use it when | Section |
|---|---|---|
| **Chaining** | Steps must happen in order, or one prompt has too many rules | 2 |
| **Routing** | Requests come in clearly different kinds | 3 |
| **Parallelization** | A job splits into independent parts | 4 |
| **Evaluator-optimizer** | You can check the result and give useful feedback | 5 |
| **Orchestrator-workers** | The parts aren't known until you see the request | 8 |
| **Agent** | The task is open-ended and can't be mapped out | 6, 7 |

**Default to the simplest thing that works.** Often that's a single well-written prompt. Then try a workflow, and reach for an agent only when the task truly needs one. Users care that the product works every time, not that it uses an agent.

Agents give you flexibility, but in exchange they:

- **cost more and vary more,** because the number of steps changes from run to run,
- **are harder to test,** because you can't list every path they might take,
- **can go wrong in more ways,** so they need turn limits, budget limits, clear tool errors, and checks on their results.

## Final spend

In [18]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

This notebook spent $0.0800 of its $0.35 budget.


## Takeaways

1. **A workflow means you choose the steps. An agent means Claude chooses them.**
2. **Chaining:** steps in order, with code checks in between, and a revise step only when needed.
3. **Routing:** classify first, then use one specialised prompt.
4. **Parallelization:** independent parts at the same time, then combine. Use a lock when parallel calls share a total.
5. **Evaluator-optimizer:** produce, check, give feedback, and always set a round limit.
6. **Agents** need small, general tools, a turn limit and a budget.
7. **Environment inspection:** read before writing, check after acting, and verify the result yourself.
8. **Orchestrator-workers:** let Claude plan the parts when you can't know them in advance.
9. **Default to workflows.** Use an agent only when the task truly needs one.

## Your turn

Each exercise costs about 1 to 3 cents.

- In section 2, add a rule that a program can check, such as "mention list comprehensions at least twice", and add it to `find_problems`.
- In section 3, add a `"comedy"` category: add it to the `Literal` and to `TEMPLATES`. Which topics end up there?
- In section 4, add a fifth criterion, `"fit with Halvora's 2024 priorities"`. What's the only other change you need?
- In section 5, lower the bullet limit to 12 words. How many rounds does it take now?
- In section 6, ask the agent a question that needs two calculations, such as comparing energy savings with plastic waste reduction.

**Next:** [12 · Claude Code and computer use](12_claude_code_and_computer_use.ipynb): Anthropic's own agents, built from the same ideas.